# Plant disease classification with transfer learning

In this project, I investigate how ImageNet pretraining affects plant disease classification when labeled training data is limited. I compare randomly initialized and ImageNet-pretrained ResNet-18 models using the same nominal 10% training subset and validation set.

I have tested the implementation on synthetic CPU data. I have not yet run the PlantVillage GPU experiment, so this notebook contains no experimental results. I reserve the published test split for final evaluation.

## 1. Set up Colab

After publishing the repository, I open this notebook through **Colab → File → Open notebook → GitHub** using `Ifethecoder/plant-disease-classifier`, and select a GPU under **Runtime → Change runtime type**. Before publication, I can upload/extract a source ZIP and set `PROJECT_DIR` to its root. I install dependencies before importing PyTorch; if libraries were already imported, I restart the runtime and rerun the setup.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd() if (Path.cwd() / "src/dataset.py").exists() else Path.cwd() / "plant-disease-classifier"
if not (PROJECT_DIR / "src/dataset.py").exists():
    subprocess.run(["git", "clone", "https://github.com/Ifethecoder/plant-disease-classifier.git",
                    str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

## 2. Choose settings and storage

I use Adam with learning rate 0.001, batch size 32 and five epochs for both models. I fine-tune all layers and use identical ImageNet normalization. I save outputs to Drive so they survive a Colab disconnection. Each run gets a new directory. With Drive disabled, results remain in temporary runtime storage and need to be downloaded before disconnecting.

In [ ]:
import json
import tempfile
from datetime import datetime, timezone

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
CACHE_DIR = Path(tempfile.gettempdir()) / "plant-village-cache"
os.environ["TORCH_HOME"] = str(CACHE_DIR / "torch")

import torch
import torchvision
from src.dataset import load_metadata, make_splits, load_images, make_loaders, seed_everything
from src.models import build_model, load_model, predict_image
from src.train import train_model
from src.evaluate import evaluate, plot_history, plot_predictions

FRACTION = 0.10  # Later experiments: 0.01, 0.03 and 0.30.
SEED = 42
EPOCHS = 5
BATCH_SIZE = 32
LEARNING_RATE = 0.001
IMAGE_SIZE = 224
VAL_FRACTION = 0.20
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    results_root = Path("/content/drive/MyDrive/plant-disease-classifier/results")
else:
    results_root = PROJECT_DIR / "results"
RUN_DIR = results_root / datetime.now(timezone.utc).strftime("demo_%Y%m%d_%H%M%S")
RUN_DIR.mkdir(parents=True)
print("Results:", RUN_DIR)

## 3. Inspect metadata and split the data

I use the pinned Hugging Face color manifests and leaf map because the repository uses a legacy dataset script. I sort class names to give both models the same label mapping.

My metadata audit found 227 leaf identifiers shared across the published splits. I exclude the 290 matching training images and preserve all 10,709 test records. I split the remaining training data by leaf group within each class, then select a fraction of the training groups. Fractions are approximate in image counts because groups have different numbers of photos. Validation stays fixed across model initializations and fractions for the same seed.

The upstream map uses filename fallbacks for some images, so physical-leaf separation cannot be guaranteed for unidentified leaves. I use test metadata only to check overlaps; I do not inspect or evaluate test images.

In [ ]:
metadata = load_metadata(CACHE_DIR)
classes = metadata["classes"]
splits = make_splits(metadata, fraction=FRACTION, seed=SEED, val_fraction=VAL_FRACTION)
print("Published splits:", metadata["published_counts"])
print("Classes:", len(classes), classes)
print("Development counts:", splits["counts"])
print("Actual image fraction:", splits["actual_image_fraction"])
(RUN_DIR / "splits.json").write_text(json.dumps(splits, indent=2))
settings = {"fraction": FRACTION, "seed": SEED, "epochs": EPOCHS,
            "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
            "image_size": IMAGE_SIZE, "val_fraction": VAL_FRACTION,
            "device": DEVICE, "dataset_revision": metadata["revision"],
            "torch_version": str(torch.__version__), "torchvision_version": torchvision.__version__,
            "source_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()}
(RUN_DIR / "settings.json").write_text(json.dumps(settings, indent=2))

## 4. Download development images

The source archive is about 2.18 GB and contains all image configurations. I cache it outside the repository and extract only the selected training and validation color images. I allow extra disk space for the extracted images and dependencies. The test images remain unopened.

In [ ]:
data = load_images(metadata, splits, CACHE_DIR)
print(data)

## 5. Train and evaluate both models

I reset the seed and rebuild the training loader for each model so both runs have the same batch order. I use full fine-tuning, cross-entropy and the same Adam settings. I select the checkpoint with the highest validation accuracy, breaking ties by validation loss.

I evaluate the selected checkpoints on the same validation set using accuracy and macro F1 across all classes. These are development scores because I also use validation for checkpoint selection. I save losses, metrics, checkpoints, prediction labels and figures.

In [ ]:
import csv

comparison = []
for name, pretrained in [("random", False), ("pretrained", True)]:
    seed_everything(SEED)
    train_loader, val_loader = make_loaders(data, splits, batch_size=BATCH_SIZE,
                                           seed=SEED, image_size=IMAGE_SIZE)
    model = build_model(len(classes), pretrained=pretrained)
    output_dir = RUN_DIR / name
    history = train_model(model, train_loader, val_loader, classes, output_dir,
                          epochs=EPOCHS, learning_rate=LEARNING_RATE,
                          device=DEVICE, image_size=IMAGE_SIZE)
    del model
    model, checkpoint = load_model(output_dir / "best.pt", DEVICE)
    metrics, actual, predicted = evaluate(model, val_loader, len(classes))
    metrics["best_epoch"] = checkpoint["epoch"]
    (output_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
    comparison.append({"model": name, **metrics})
    plot_history(history, output_dir / "learning_curves.png")
    plot_predictions(val_loader.dataset, actual, predicted, classes, output_dir / "examples.png")
    has_mistakes = plot_predictions(val_loader.dataset, actual, predicted, classes,
                                   output_dir / "mistakes.png", mistakes_only=True)
    if not has_mistakes:
        (output_dir / "mistakes.txt").write_text("No validation misclassifications in this run.")
    with (output_dir / "predictions.csv").open("w", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["image_path", "actual", "predicted"])
        writer.writerows(zip(val_loader.dataset.records["image_path"],
                             [classes[i] for i in actual], [classes[i] for i in predicted]))
    del model

with (RUN_DIR / "comparison.csv").open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(comparison[0]))
    writer.writeheader()
    writer.writerows(comparison)

## 6. Review results

I compare the observed scores and learning curves only after training completes. I examine both correct predictions and mistakes, and discuss class imbalance, the short training budget and the limitations of controlled-background images. I do not infer field-image reliability from PlantVillage validation scores.

In [ ]:
from IPython.display import display, Markdown, Image as DisplayImage

table = "| Model | Accuracy | Macro F1 | Best epoch |\n|---|---:|---:|---:|\n"
for row in comparison:
    table += f"| {row['model']} | {row['accuracy']:.4f} | {row['macro_f1']:.4f} | {row['best_epoch']} |\n"
(RUN_DIR / "comparison.md").write_text(table)
display(Markdown(table))
for name in ("random", "pretrained"):
    display(Markdown(f"### {name.title()} ResNet-18"))
    for filename in ("learning_curves.png", "examples.png", "mistakes.png"):
        path = RUN_DIR / name / filename
        if path.exists():
            display(DisplayImage(filename=str(path)))
    note = RUN_DIR / name / "mistakes.txt"
    if note.exists():
        print(note.read_text())

## 7. Try inference

I demonstrate inference on a validation image, showing its actual class alongside the prediction. I restore the label mapping and image size from the checkpoint. Confidence is a softmax score, not a calibrated probability. I can also pass a local image path to `predict_image`.

In [ ]:
model, checkpoint = load_model(RUN_DIR / "pretrained/best.pt", DEVICE)
example = val_loader.dataset.records[0]
display(example["image"])
print("Actual:", classes[example["label"]])
print("Prediction:", predict_image(example["image"], model, checkpoint["classes"], checkpoint["image_size"]))

## Next experiments

I plan to repeat the comparison with 1%, 3%, 10% and 30% of training leaf groups and multiple seeds. I will keep validation fixed across fractions for each seed and reserve test evaluation for the final protocol. Classifier placement experiments are deferred.

I retain the complete run directory in Drive for my progress report. It contains the settings, split indices, two best checkpoints, losses, validation metrics and figures. If a run is interrupted, the completed histories/checkpoints remain available, but automatic resume is not implemented. I start a fresh run directory when repeating training.